In [ ]:
# ============================================================
# 01_ENARES_2024_STAGE2_bigquery_setup.ipynb
# Stage 2 - BigQuery setup
# Objective: create/verify raw, cleaned, and analytical datasets
# ============================================================

!pip install -q google-cloud-bigquery pandas pyreadstat pandas-gbq

from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
import pandas as pd
import os

auth.authenticate_user()
drive.mount("/content/drive")

PROJECT_ID = input("Enter your Google Cloud PROJECT_ID: ").strip()
LOCATION = "US"

ROOT_DRIVE = "/content/drive/MyDrive/ENARES_2024_PROJECT"
LOG_DIR = f"{ROOT_DRIVE}/05Resultados/logs"

os.makedirs(LOG_DIR, exist_ok=True)

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

# Simple BigQuery connection test
test_query = client.query("SELECT CURRENT_DATE() AS current_date")
display(test_query.result().to_dataframe())

In [ ]:
# ============================================================
# Create or verify required BigQuery datasets
# ============================================================

datasets = [
    "enares2024_crs04_raw",
    "enares2024_crs04_cleaned",
    "enares2024_crs04_analytical",
]

registry = []

for dataset_name in datasets:
    dataset_id = f"{PROJECT_ID}.{dataset_name}"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = LOCATION

    try:
        client.create_dataset(dataset, exists_ok=True)
        verified_dataset = client.get_dataset(dataset_id)

        status = "created_or_exists"
        error_message = None

        registry.append({
            "project_id": PROJECT_ID,
            "dataset_id": dataset_name,
            "full_dataset_id": dataset_id,
            "location": verified_dataset.location,
            "status": status,
            "error_message": error_message,
            "checked_at_utc": datetime.now(timezone.utc).isoformat()
        })

    except Exception as e:
        registry.append({
            "project_id": PROJECT_ID,
            "dataset_id": dataset_name,
            "full_dataset_id": dataset_id,
            "location": LOCATION,
            "status": "error",
            "error_message": str(e),
            "checked_at_utc": datetime.now(timezone.utc).isoformat()
        })

dataset_registry = pd.DataFrame(registry)

output_path = f"{LOG_DIR}/ENARES_2024_STAGE2_bigquery_dataset_registry.csv"
dataset_registry.to_csv(output_path, index=False)

print(f"Output saved to: {output_path}")
display(dataset_registry)

In [ ]:
# ============================================================
# Create or verify required BigQuery datasets
# ============================================================

datasets = [
    "enares2024_crs04_raw",
    "enares2024_crs04_cleaned",
    "enares2024_crs04_analytical",
]

registry = []

for dataset_name in datasets:
    dataset_id = f"{PROJECT_ID}.{dataset_name}"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = LOCATION

    try:
        client.create_dataset(dataset, exists_ok=True)
        verified_dataset = client.get_dataset(dataset_id)

        status = "created_or_exists"
        error_message = None

        registry.append({
            "project_id": PROJECT_ID,
            "dataset_id": dataset_name,
            "full_dataset_id": dataset_id,
            "location": verified_dataset.location,
            "status": status,
            "error_message": error_message,
            "checked_at_utc": datetime.now(timezone.utc).isoformat()
        })

    except Exception as e:
        registry.append({
            "project_id": PROJECT_ID,
            "dataset_id": dataset_name,
            "full_dataset_id": dataset_id,
            "location": LOCATION,
            "status": "error",
            "error_message": str(e),
            "checked_at_utc": datetime.now(timezone.utc).isoformat()
        })

dataset_registry = pd.DataFrame(registry)

output_path = f"{LOG_DIR}/ENARES_2024_STAGE2_bigquery_dataset_registry.csv"
dataset_registry.to_csv(output_path, index=False)

print(f"Output saved to: {output_path}")
display(dataset_registry)

In [ ]:
# ============================================================
# Acceptance check
# ============================================================

if not (dataset_registry["status"] == "created_or_exists").all():
    display(dataset_registry)
    raise RuntimeError("One or more BigQuery datasets could not be created or verified.")

print("Notebook 1 completed successfully.")
print("Required output created:")
print("ENARES_2024_STAGE2_bigquery_dataset_registry.csv")